In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import re

# 🔽 Ваши импорты
from readerscf import parse_sdr_file
from get_project_root import get_project_root
from substractdf import subtract_column_min
from center_dataframe import center_dataframe
from estimate_M_from_data import estimate_M_from_data
from estimate_crosstalk_matrix import estimate_crosstalk_matrix
from estimate_M_correlation import estimate_M_correlation,estimate_M_correlation_crostalk
from deconvolve_domnisoru import deconvolve_domnisoru
from multiply_matrix_with_dataframe import multiply_matrix_with_dataframe
from estimate_M_goodpeaks_crostalk import estimate_M_goodpeaks_crostalk
from estimate_M_clusters_crostalk import estimate_M_clusters_crostalk
from estimate_M_from_clean_peaks import estimate_M_from_clean_peaks
from divide_matrices_np import divide_matrices_np
from normalize_diagonal import normalize_diagonal
from itertools import combinations
from config import IUPAC, ref_str, color_map
from multiply_matrix_with_dataframe import multiply_matrix_with_dataframe,matrix_multiply_with_dataframe

def sanitize_sheet_name(filename: str) -> str:
    """Приводит имя файла к формату, допустимому для листа Excel (макс 31 символ, без спецсимволов)."""
    name = Path(filename).stem  # убираем .srd
    name = re.sub(r'[\\\/\?\*\[\]:]', '', name)
    if len(name) > 31:
        name = name[:28] + "..."
    return name

def save_matrices_to_excel(results: list, output_path: Path):
    """Сохраняет 4 матрицы для каждого файла в отдельные листы одного Excel-файла."""
    if not results:
        print("⚠️ Нет данных для сохранения.")
        return

    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        used_names = {}
        
        for res in results:
            # Уникальное имя листа
            base_name = sanitize_sheet_name(res['file'])
            if base_name in used_names:
                used_names[base_name] += 1
                sheet_name = f"{base_name}_{used_names[base_name]}"
            else:
                used_names[base_name] = 0
                sheet_name = base_name

            start_row = 0
            matrices = [
                ('matrixdef',    '🔹 M0: Default'),
                ('matrix1',    '🔹 M1: Estimated'),
                ('matrix2',    '🔹 M2: Crosstalk'),
                ('matrix3',    '🔹 M3: Crosstalk correlation'),
                ('matrix4',    '🔹 M4: Crosstalk cluster'),
                ('matrix5',    '🔹 M5: Crosstalk good peaks'),
              
            ]

            for mat_key, title in matrices:
                df = pd.DataFrame(res[mat_key])
                df.index = [f'Row {i+1}' for i in range(df.shape[0])]
                df.columns = [f'Col {j+1}' for j in range(df.shape[1])]
                
                # Заголовок матрицы
                pd.DataFrame([[title]]).to_excel(writer, sheet_name=sheet_name, 
                                                 startrow=start_row, index=False, header=False)
                # Сама матрица (через 1 строку после заголовка)
                df.to_excel(writer, sheet_name=sheet_name, 
                            startrow=start_row + 2, index=True, header=True)
                
                start_row += df.shape[0] + 4  # данные + заголовок + отступ + заголовок следующей

            tqdm.write(f"   📄 Лист '{sheet_name}' сохранён")

    print(f"💾 Итоговый файл: {output_path}")

import matplotlib.pyplot as plt
from pathlib import Path

def save_pairs_plot(data, matrix_name: str,folder_name:str, file_name: str, project_root: Path):
    # Получаем все пары колонок (без повторов)
    pairs = list(combinations(data.columns, 2))

    # Рассчитываем сетку subplots
    n_pairs = len(pairs)
    n_cols = 3  # Число столбцов в сетке
    n_rows = (n_pairs + n_cols - 1) // n_cols  # Округление вверх
    """Отрисовывает scatter-матрицу и сохраняет с динамическим именем."""
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 8))
    fig.suptitle(f"Попарные scatter plots после {matrix_name}", fontsize=14)

    for idx, (x_col, y_col) in enumerate(pairs):
        ax = axes.flatten()[idx]
        ax.scatter(data[x_col], data[y_col], alpha=0.7, color=[color_map[x_col]])
        ax.set_xlabel(x_col)
        ax.set_ylabel(y_col)
        ax.grid(True, linestyle='--', alpha=0.5)

    for idx in range(n_pairs, n_rows * n_cols):
        fig.delaxes(axes.flatten()[idx])

    plt.tight_layout()

    # 🔹 Динамический путь: имя метода/pairsplotafter{имя_матрицы}_{имя_файла}.jpeg
    save_dir = Path(project_root) / "results"/ folder_name
    save_dir.mkdir(parents=True, exist_ok=True)
    save_path = save_dir / f"pairsplotafter{matrix_name}_{Path(file_name).stem}.jpeg"

    plt.savefig(save_path, dpi=300)
    plt.close(fig)  # 🔥 КРИТИЧНО: иначе память заполнится и скрипт упадёт на 10-20 файле


def main():
    project_root = Path(get_project_root())
    print(f"📁 Корень проекта: {project_root}")

    srd_dir = project_root / "files" / "сырые SRD" / "сырые SRD"
    if not srd_dir.is_dir():
        print(f"❌ Папка не найдена: {srd_dir}")
        return

    srd_files = sorted(srd_dir.glob("*.srd"))
    if not srd_files:
        print("⚠️ Файлы .srd не найдены.")
        return

    print(f"📂 Найдено файлов: {len(srd_files)}\n")

    results = []  # 📦 Накопитель результатов

    with tqdm(srd_files, desc="🔄 Обработка SRD", unit="файл") as pbar:
        for srd_path in pbar:
            pbar.set_postfix(file=srd_path.name[:18] + "..." if len(srd_path.name) > 18 else srd_path.name)
            
            try:
                sdr_matrix, sdr_channels, sdr_meta =parse_sdr_file(srd_path)
                matrix = sdr_matrix.to_numpy()
                
                if matrix.shape[0] < 10 or matrix.shape[1] < 4:
                    tqdm.write(f"   ⚠️ Матрица слишком мала, пропуск.")
                    continue
                    
                matrixdef = matrix[6:10, 0:4]
                
                data0 = sdr_channels.loc[:, ['dR110', 'dR6G', 'dTAMRA', 'dROX']].copy()
                data0.columns = ['G', 'A', 'T', 'C']
                data0 = subtract_column_min(data0)
                data0 = center_dataframe(data0, method='percentile', percentile=2)

                data1 = multiply_matrix_with_dataframe(matrixdef, data0)
                save_pairs_plot(data1, "matrixdef","defaultcallibration", srd_path.name, project_root)
                
                matrix1 = estimate_M_from_data(
                    raw=data0, dye_order=['G', 'A', 'T', 'C'],
                    min_purity=0.5, peak_height=150,
                    peak_distance=15, peak_prominence=80
                )
                data1 = multiply_matrix_with_dataframe(matrix1, data0)
                save_pairs_plot(data1, "matrix1", "estimate_M_from_data",srd_path.name, project_root)
                matrix2 = estimate_crosstalk_matrix(data0, init_M=matrixdef)
                data1 = multiply_matrix_with_dataframe(matrix2, data0)
                save_pairs_plot(data1, "matrix2","estimate_crosstalk_matrix", srd_path.name, project_root)
                matrix3=estimate_M_correlation_crostalk(data0,init_M=matrixdef)
                data1 = multiply_matrix_with_dataframe(matrix3, data0)
                save_pairs_plot(data1, "matrix3","estimate_M_correlation_crostalk", srd_path.name, project_root)
                matrix4=estimate_M_clusters_crostalk(data0)
                data1 = multiply_matrix_with_dataframe(matrix4, data0)
                save_pairs_plot(data1, "matrix4", "estimate_M_clusters_crostalk",srd_path.name, project_root)
                matrix5=estimate_M_goodpeaks_crostalk(data0)
                data1 = multiply_matrix_with_dataframe(matrix5, data0)
                save_pairs_plot(data1, "matrix5", "estimate_M_goodpeaks_crostalk",srd_path.name, project_root)
                matrix1=normalize_diagonal(matrix1)
                matrix2=normalize_diagonal(matrix2)
                matrix3=normalize_diagonal(matrix3)
                matrix4=normalize_diagonal(matrix4)
                matrix5=normalize_diagonal(matrix5)
                
               

                # 📥 Сохраняем в память
                results.append({
                    'file': srd_path.name,
                    'matrixdef': matrixdef, 
                    'matrix1': matrix1, 'matrix2': matrix2,'matrix3':matrix3,'matrix4':matrix4,'matrix5':matrix5,
                    
                })
                
            except Exception as e:
                tqdm.write(f"   ❌ Ошибка в {srd_path.name}: {e}")

    # 💾 Запись в Excel (папка result)
    output_dir = project_root / "results"/"excel"
    output_dir.mkdir(parents=True, exist_ok=True)
    output_file = output_dir / "srd_all_matrices.xlsx"
    save_matrices_to_excel(results, output_file)


if __name__ == "__main__":
    main()

📁 Корень проекта: C:\Users\Admin\Documents\GitHub\dnasegnercrosstalk
📂 Найдено файлов: 40



🔄 Обработка SRD:   0%|                                           | 0/40 [00:00<?, ?файл/s, file=2_pGEM_G2_PDMA6_36...]

Найдено пиков: 593
  Итерация 1: max Δ = 0.615097
  Итерация 2: max Δ = 0.042758
  Итерация 3: max Δ = 0.004301
  Сходимость на итерации 4
Найдено пиков: 592
  Итерация 1: max Δ = 0.590666
  Итерация 2: max Δ = 0.128326
  Итерация 3: max Δ = 0.435195
  Итерация 6: max Δ = 0.120083
  Итерация 11: max Δ = 0.000000
  Сходимость на итерации 11
Найдено пиков: 592
  Итерация 1: max Δ = 11.812703
  Итерация 2: max Δ = 0.190016
  Итерация 3: max Δ = 0.043610
  Итерация 6: max Δ = 0.114232
  Итерация 11: max Δ = 0.121691
  Итерация 16: max Δ = 0.121691
  Итерация 21: max Δ = 0.121691
  Итерация 26: max Δ = 0.121691
Найдено пиков: 592
  Итерация 1: max Δ = 0.195740
  Итерация 2: max Δ = 7.040791
  Итерация 3: max Δ = 7.188878
  Итерация 6: max Δ = 0.005797
  Сходимость на итерации 8


🔄 Обработка SRD:   2%|▉                                  | 1/40 [00:09<06:19,  9.73s/файл, file=3_pGEM_A3_PDMA6_50...]

Найдено пиков: 876
  Итерация 1: max Δ = 0.612441
  Итерация 2: max Δ = 0.075450
  Итерация 3: max Δ = 1.125717
  Итерация 6: max Δ = 2.007002
  Итерация 11: max Δ = 1.697462
  Итерация 16: max Δ = 1.697462
  Итерация 21: max Δ = 1.697462
  Итерация 26: max Δ = 1.697462
Найдено пиков: 871
  Итерация 1: max Δ = 1.246055
  Итерация 2: max Δ = 0.099299
  Итерация 3: max Δ = 0.030549
  Итерация 6: max Δ = 1.738466
  Сходимость на итерации 9
Найдено пиков: 871
  Итерация 1: max Δ = 9.027085
  Итерация 2: max Δ = 0.154080
  Итерация 3: max Δ = 0.128717
  Итерация 6: max Δ = 0.011122
  Итерация 11: max Δ = 0.004258
  Итерация 16: max Δ = 0.004258
  Итерация 21: max Δ = 0.004258
  Итерация 26: max Δ = 0.004258
Найдено пиков: 871
  Итерация 1: max Δ = 0.181881
  Итерация 2: max Δ = 0.166661
  Итерация 3: max Δ = 0.045751
  Итерация 6: max Δ = 0.000747
  Сходимость на итерации 7


...

Найдено пиков: 534
  Итерация 1: max Δ = 0.627152
  Итерация 2: max Δ = 0.018095
  Итерация 3: max Δ = 0.002091
  Сходимость на итерации 5
Найдено пиков: 534
  Итерация 1: max Δ = 0.274243
  Итерация 2: max Δ = 0.278671
  Итерация 3: max Δ = 0.047055
  Сходимость на итерации 5
Найдено пиков: 534
  Итерация 1: max Δ = 1.920118
  Итерация 2: max Δ = 0.297056
  Итерация 3: max Δ = 0.262957
  Итерация 6: max Δ = 0.341121
  Итерация 11: max Δ = 0.079470
  Итерация 16: max Δ = 0.088763
  Итерация 21: max Δ = 0.088763
  Итерация 26: max Δ = 0.088763
Найдено пиков: 534
  Итерация 1: max Δ = 0.135403
  Итерация 2: max Δ = 0.289032
  Итерация 3: max Δ = 0.033229
  Итерация 6: max Δ = 0.004962
  Сходимость на итерации 8


🔄 Обработка SRD:  95%|████████████████████████████████▎ | 38/40 [07:41<00:21, 10.91s/файл, file=pGEM_4_G2_PDMA6_36...]

Найдено пиков: 545
  Итерация 1: max Δ = 0.618600
  Итерация 2: max Δ = 0.016509
  Итерация 3: max Δ = 0.001502
  Сходимость на итерации 4
Найдено пиков: 545
  Итерация 1: max Δ = 0.144234
  Итерация 2: max Δ = 0.122221
  Итерация 3: max Δ = 0.025174
  Сходимость на итерации 5
Найдено пиков: 545
  Итерация 1: max Δ = 13.090853
  Итерация 2: max Δ = 0.968544
  Итерация 3: max Δ = 0.649436
  Итерация 6: max Δ = 0.000701
  Сходимость на итерации 7
Найдено пиков: 545
  Итерация 1: max Δ = 0.184518
  Итерация 2: max Δ = 0.960350
  Итерация 3: max Δ = 10.994270
  Итерация 6: max Δ = 0.003673
  Сходимость на итерации 7


🔄 Обработка SRD:  98%|█████████████████████████████████▏| 39/40 [07:51<00:10, 10.69s/файл, file=pGEM_4_H2_PDMA6_36...]

Найдено пиков: 525
  Итерация 1: max Δ = 0.623409
  Итерация 2: max Δ = 0.011424
  Итерация 3: max Δ = 0.000774
  Сходимость на итерации 4
Найдено пиков: 525
  Итерация 1: max Δ = 0.145991
  Итерация 2: max Δ = 0.063865
  Итерация 3: max Δ = 0.012331
  Сходимость на итерации 5
Найдено пиков: 525
  Итерация 1: max Δ = 2.467182
  Итерация 2: max Δ = 1.274323
  Итерация 3: max Δ = 0.065455
  Сходимость на итерации 5
Найдено пиков: 525
  Итерация 1: max Δ = 0.326151
  Итерация 2: max Δ = 0.078743
  Итерация 3: max Δ = 0.016960
  Итерация 6: max Δ = 0.000000
  Сходимость на итерации 6
